---
title: "Chương 3: Tổ chức dữ liệu và kết quả discovery"
short_title: "3. Tổ chức dữ liệu và kết quả discovery"
---

## 1. Cấu trúc corpus làm việc

Mỗi khoảng ID (của một thành viên) có một thư mục dữ liệu (`--data-dir`) trên ổ của runtime Colab (`/content/g11/<tên>_<ID đầu>_<ID cuối>`), nằm ngoài repository code. Sau mỗi bước crawl, phần thay đổi được nén thành một file `.tar.gz` lưu lên Google Drive làm bản lưu; runtime mới khôi phục thư mục từ các bản lưu đó rồi chạy tiếp. Bên trong, mỗi bài báo có một thư mục riêng tên `<yymm-id>` (ví dụ `2604-05535` cho ID `2604.05535`), đúng quy ước gợi ý của đề:

```text
<data-dir>/
├── work/
│   └── paper_manifest.jsonl     một dòng mỗi ID: trạng thái discovery, danh sách phiên bản
├── corpus/<yymm-id>/
│   ├── metadata.json            metadata của bài và của từng phiên bản
│   ├── versions.json            trạng thái tải/trích xuất của từng phiên bản
│   ├── v1/, v2/, ...            các file .tex/.bib, giữ nguyên thư mục con của gói nguồn
│   ├── references_all.json      toàn bộ reference từ Semantic Scholar
│   └── references.json          chỉ các reference có arXiv ID
└── logs/                        log CSV phục vụ đo đạc và báo cáo
```

Phiên bản PDF-only hoặc không có TeX vẫn có thư mục `v<N>/` (rỗng) và một bản ghi trạng thái trong `versions.json`, nên không bài báo nào bị bỏ khỏi corpus.

## 2. Nội dung các file

| File | Trường chính |
| :--- | :--- |
| `metadata.json` | `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue`, `primary_category`, `categories`, `versions[]` (metadata từng phiên bản) |
| `versions.json` | Theo từng phiên bản: `status`, `source_format`, `downloaded_bytes`, `original_bytes`, `image_bytes`, `kept_bytes`, `kept_files`, `attempts`, `finished_at` |
| `references_all.json` | `status`, `s2_paper_id`, `s2_reference_count`, `arxiv_dates_complete`, `references[]` với `ref_arxiv_id`, `ref_title`, `ref_authors`, `ref_submission_date`, `ref_s2_id`, `metadata_status`, … |
| `references.json` | Khóa là arXiv ID được trích dẫn (dạng `yymm-id`): `title`, `authors`, `submission_date`, `semantic_scholar_id` |

## 3. Ánh xạ sang các bảng của dataset cuối

Đề quy định hai bảng sẽ được công bố ở Milestone 3. Corpus làm việc đã có đủ dữ liệu để tạo chúng mà không phải crawl lại:

| Bảng | Cột | Lấy từ |
| :--- | :--- | :--- |
| `papers` | `arxiv_id`, `title`, `authors`, `submission_date`, `revised_dates`, `venue` | Các trường cùng tên ở cấp ngoài cùng của `metadata.json` |
| `references` | `arxiv_id`, `ref_arxiv_id`, `ref_title`, `ref_authors`, `ref_submission_date`, `ref_s2_id` | Các reference có `ref_arxiv_id` trong `references_all.json` |

## 4. Mã trạng thái

| Bước | Trạng thái | Ý nghĩa |
| :--- | :--- | :--- |
| Discovery | `success` / `not_found` / `api_error` | ID tồn tại / arXiv không có ID này / API lỗi, thử lại ở lượt sau |
| Phiên bản | `success` / `pdf_only` / `no_tex` / `not_found` / `failed` | Có `.tex`/`.bib` / chỉ có PDF / DOCX hoặc PostScript / arXiv trả 404 / lỗi tạm thời, thử lại |
| References | `success` / `empty` / `unavailable` / `not_found` / `api_error` | Xem [Chương 2](./02_methodology.ipynb), mục 6 |

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, pct, provenance, show, table)
provenance()

## 5. Khoảng ID của từng thành viên

Số ID được đọc từ manifest của từng thư mục dữ liệu.

In [ ]:
m = members()
ranges = pd.DataFrame({
    "Thành viên": m["member"], "ID đầu": m["first_id"], "ID cuối": m["last_id"],
    "Số ID": m["ids"], "Bài tồn tại": m["papers"], "Không tồn tại": m["ids_not_found"],
    "Lỗi API": m["ids_api_error"], "Số phiên bản": m["versions"]})
g = group()
ranges.loc[len(ranges)] = ["Cả nhóm", g["first_id"], g["last_id"], g["ids"], g["papers"],
                           g["ids_not_found"], g["ids_api_error"], g["versions"]]
table(ranges, "Khoảng ID và kết quả discovery theo thành viên")

## 6. Số phiên bản của mỗi bài

Đề yêu cầu crawl **mọi phiên bản**; phân bố dưới đây cho biết bao nhiêu bài có nhiều hơn một phiên bản.

In [ ]:
papers = load("papers")
found = papers[papers["discovery_status"] == "success"]
dist = found.groupby("n_versions").size().reset_index(name="papers")
dist["share"] = dist["papers"] / dist["papers"].sum()
fig = px.bar(dist, x="n_versions", y="papers", text=dist["share"].map(pct),
             color_discrete_sequence=COLORS,
             title=f"Số phiên bản của mỗi bài ({len(found):,} bài tồn tại)",
             labels={"n_versions": "Số phiên bản", "papers": "Số bài"})
fig.update_xaxes(dtick=1)
show(fig)
print(f"Trung bình {found['n_versions'].mean():.3f} phiên bản/bài; "
      f"{(found['n_versions'] > 1).mean():.2%} số bài có từ 2 phiên bản trở lên.")

## 7. Lĩnh vực của các bài báo

Lĩnh vực chính (`primary_category`) lấy từ metadata arXiv.

In [ ]:
cats = found["primary_category"].fillna("(không có)")
top = cats.value_counts().head(20).rename_axis("category").reset_index(name="papers")
fig = px.bar(top.iloc[::-1], x="papers", y="category", orientation="h", color_discrete_sequence=COLORS,
             title="20 lĩnh vực chính có nhiều bài nhất",
             labels={"papers": "Số bài", "category": "primary_category"})
fig.update_layout(height=600)
show(fig)
archive = cats.str.split(".").str[0].value_counts().rename_axis("archive").reset_index(name="papers")
fig = px.pie(archive, names="archive", values="papers", color_discrete_sequence=COLORS,
             title="Tỷ lệ bài theo nhóm lĩnh vực (phần trước dấu chấm của primary_category)")
show(fig)